# wymのExplanation結果を作成する


```bash
PYTHONPATH=/workspaces/entity-matching-explainer/wym:/workspaces/entity-matching-explainer \
    TARGET_DATASET_ID=1 TOP_N=5 \
    nohup bash -c 'papermill  \
              --prepare-execute --log-output \
              -p TARGET_DATASET_ID ${TARGET_DATASET_ID} -p TOP_N ${TOP_N}\
              eval_wym.ipynb eval_wym_${TARGET_DATASET_ID}_${TOP_N}.ipynb \
              >eval_wym_${TARGET_DATASET_ID}_${TOP_N}.log 2>&1 &'
```


In [ ]:
matcher_names = ["wym"]
dataset_root_dir = "../../data/lemon/datasets"
model_root_dir = "../../data/wym/model"
wym_result_root_dir = "../../data/experiments/11_eval/wym_result"
out_root_dir = "../../data/experiments/11_eval/wym"
out_root_dir_no_intercept = None
dataset_names = [
    "structured_amazon_google",
    "structured_beer",
    "structured_dblp_acm",
    "structured_dblp_google_scholar",
    "structured_fodors_zagat",
    "structured_walmart_amazon",
    "structured_itunes_amazon",
    "dirty_dblp_acm",
    "dirty_dblp_google_scholar",
    "dirty_walmart_amazon",
    "dirty_itunes_amazon",
    "textual_abt_buy",
    "textual_company",
]

In [ ]:
TARGET_DATASET_ID = 1
TOP_N = 5
GPU_ID = 0
# LIME_NO_INTERCEPT = False

In [ ]:
BATCH_SIZE = 512
gpu_id = GPU_ID

In [ ]:
# スレッド数を制限
## これをしないと、他のプロセスが利用するCPUがなくなってしまう
import os

THREAD_NUM = 5

# NumPy (OpenBLAS, MKL 等) が使用するスレッド数を制限
os.environ["OMP_NUM_THREADS"] = str(THREAD_NUM)
os.environ["MKL_NUM_THREADS"] = str(THREAD_NUM)
os.environ["OPENBLAS_NUM_THREADS"] = str(THREAD_NUM)

import numpy as np
import torch

# PyTorch のスレッド数制限 (演算用)
torch.set_num_threads(THREAD_NUM)
# PyTorch のスレッド数制限 (DataLoader 等のインタロップ用)
torch.set_num_interop_threads(THREAD_NUM)

In [ ]:
# torchモジュールの読み込み前に、利用できるGPUを指定しておく
## これをやらないと、システム内のＧＰＵすべてを利用してしまう
import os

os.environ["CUDA_VISIBLE_DEVICES"] = f"{GPU_ID}"

import torch

print("CUDA =", torch.cuda.is_available())
print("CUDA DEVICES =", torch.cuda.device_count())
print("CUDA CURRENT DEVICE_ID = ", torch.cuda.current_device())

In [ ]:
# transformers の tokenizer を並列実行で呼び出すか（dead lockしてしまう）
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
# Set Random Seeds and Reproducibility
import random

import numpy as np


def set_seed(seed: int):
    """
    Helper function for reproducible behavior to set the seed in ``random``, ``numpy``, ``torch``
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(0)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

## データセットを読み込む

In [ ]:
from pine.dataset import load_dataset
from lemon.utils.datasets import SplittedDataset


def test_load_dataset():
    dataset_name = dataset_names[TARGET_DATASET_ID]
    dataset: SplittedDataset = load_dataset(dataset_name, dataset_root_dir)
    display(dataset.test.records.a.head())
    display(dataset.test.records.b.head())
    display(dataset.test.record_id_pairs.head())
    display(dataset.test.labels.head())
    print(dataset.test.records.a.dtypes)
    print("Test DATA SIZE =", len(dataset.test.record_id_pairs))


test_load_dataset()

### マッチスコア出力関数

### データセット変換

In [ ]:
import pandas as pd
import unicodedata


def remove_accents(input_str: str) -> str:
    """
    Unicode のアクセント記号を削除して基本的な a-z に変換
    Args:
        input_str (str): 入力文字列
    Returns:
        str: アクセント記号を削除した文字列
    """
    # Unicode 正規化 (NFKD) によって分解
    normalized_str = unicodedata.normalize("NFKD", input_str)
    # 分解されたアクセント記号を除去
    return "".join(c for c in normalized_str if not unicodedata.combining(c))


def normalize_str(df: pd.DataFrame) -> pd.DataFrame:
    """
    文字列の正規化を行う。ウムラウト系を削除。大文字小文字を統一する。
    Args:
        df (pd.DataFrame): 文字列を含むDataFrame
    Returns:
        pd.DataFrame: 正規化されたDataFrame
    """

    # 置換関数を定義
    def replace_chars(value):
        if isinstance(value, str):  # 文字列のみ処理
            value = remove_accents(value).lower()
        return value

    # 各列ごとに処理し、元の型を維持
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].apply(replace_chars).astype("string")
        else:
            df[col] = df[col]  # 非文字列型はそのまま
    return df


def convert_dataset_to_wym(
    records_left: pd.DataFrame,
    records_right: pd.DataFrame,
    record_id_pairs: pd.DataFrame,
    labels: pd.Series = None,
) -> pd.DataFrame:
    """
    lemonのデータセットをWYMの入力形式に変換する
    WWMの入力形式は以下のDatarrameを返す。
    id,left_id,right_id,label,left_*(左側のカラム), right_右側のカラム

    Args:
        records_left (pd.DataFrame): 左側のレコード
        records_right (pd.DataFrame): 右側のレコード
        record_id_pairs (pd.DataFrame): レコードIDのペア
        labels (pd.DataFrame, optional): ラベル. Defaults to None.
    Returns:
        pd.DataFrame: WYMの入力形式
    """
    # record_leftとrecord_rightの内容のうち文字列を変更する
    records_left = normalize_str(records_left)
    records_right = normalize_str(records_right)

    # カラム名を変更
    df = record_id_pairs.rename(columns={"a.rid": "left_id", "b.rid": "right_id"})
    # ラベルがあれば結合
    if labels is not None:
        df = pd.merge(df, labels.astype(int), left_index=True, right_index=True)

    # レコードIDのペアを結合
    df = pd.merge(
        df,
        records_left.add_prefix("left_"),
        left_on="left_id",
        right_index=True,
    )
    df = pd.merge(
        df, records_right.add_prefix("right_"), left_on="right_id", right_index=True
    )

    # インデックスをリセットしpidをidに変換する
    df = df.sort_index().reset_index().rename(columns={"pid": "id"})

    return df


def test_convert_dataset_to_wym():
    dataset_name = dataset_names[TARGET_DATASET_ID]
    dataset: SplittedDataset = load_dataset(dataset_name, dataset_root_dir)
    df = convert_dataset_to_wym(
        dataset.test.records.a,
        dataset.test.records.b,
        dataset.test.record_id_pairs,
        dataset.test.labels,
    )
    display(df.head())
    print("DATA SIZE =", len(df))
    print(df.dtypes)


test_convert_dataset_to_wym()


In [ ]:
from typing import Callable, List

from pine.entity import EntityPair, Entity


def convert_entity_pairs_to_wym(entity_pairs: List[EntityPair]) -> pd.DataFrame:
    """
    EntityPair形式をWYMの入力形式に変換する

    Args:
        entity_pairs (List[EntityPair]): EntityPairのリスト
    Returns:
        pd.DataFrame: WYMの入力形式
    """
    df_lefts = []
    df_rights = []
    df_pairs = []
    for idx, entity_pair in enumerate(entity_pairs):
        df_left, df_right = entity_pair.to_dataframe()
        # idを振りなおしながら追加
        df_left.index = [idx]
        df_right.index = [len(entity_pairs) + idx]
        df_lefts.append(df_left)
        df_rights.append(df_right)
        df_pairs.append(
            pd.DataFrame(
                {
                    "pid": [idx],
                    "a.rid": [df_left.index[0]],
                    "b.rid": [df_right.index[0]],
                }
            )
        )
    df_lefts = pd.concat(df_lefts)
    df_rights = pd.concat(df_rights)
    df_pairs = pd.concat(df_pairs).set_index("pid")
    record_pair = convert_dataset_to_wym(df_lefts, df_rights, df_pairs)
    return record_pair


def test_convert_entity_pairs_to_wym():
    dataset_name = dataset_names[TARGET_DATASET_ID]
    dataset: SplittedDataset = load_dataset(dataset_name, dataset_root_dir)

    entity_pairs = []
    for pid, l_id, r_id in dataset.train.record_id_pairs[:10].itertuples():
        entity_pair = EntityPair(
            Entity.from_dataframe(dataset.train.records.a.loc[[l_id]]),
            Entity.from_dataframe(dataset.train.records.b.loc[[r_id]]),
        )
        entity_pairs.append(entity_pair)
    df = convert_entity_pairs_to_wym(entity_pairs)
    display(df)
    print("DATA SIZE =", len(df))
    print(df.dtypes)
    assert len(entity_pairs) == 10
    return


test_convert_entity_pairs_to_wym()


In [ ]:
from typing import Callable, List
import os
import pickle


import torch
from wym.wym import Wym
from wym.Net import NetAccoppiate, DatasetAccoppiate
from pine.entity import EntityPair, Entity

import warnings

# SettingWithCopyWarning を無視する設定
#warnings.simplefilter(action="ignore", category=FutureWarning)
#warnings.simplefilter(
#    action="ignore", category=UserWarning
#)  # これで SettingWithCopyWarning を無視


# モンキーパッチ
## メソッドの変更
def new_relevance_score(self, word_pairs, emb_pairs):
    self.word_pair_model.eval()
    self.word_pair_model.to(self.device)
    # data_loader = self.train_data_loader
    # data_loader.__init__(word_pairs, emb_pairs)
    data_loader = DatasetAccoppiate(word_pairs, emb_pairs)
    word_pair_corrected = data_loader.word_pairs_corrected
    with torch.no_grad():
        word_pair_corrected["pred"] = (
            self.word_pair_model(data_loader.X.to(self.device)).cpu().detach().numpy()
        )
    return word_pair_corrected


Wym.relevance_score = new_relevance_score


## メンバー変数の変更
def modify_wym_member(wym: Wym) -> Wym:
    # word_pair_model の追加
    tmp_path = os.path.join(wym.model_files_path, "net.pickle")
    best_model = NetAccoppiate()
    best_model.load_state_dict(
        torch.load(tmp_path, map_location=torch.device(wym.device))
    )
    wym.word_pair_model = best_model

    # best_linear_model_data の追加
    tmp_path = os.path.join(wym.model_files_path, "linear_model.pickle")
    with open(tmp_path, "rb") as file:
        model_data = pickle.load(file)
    wym.best_linear_model_data = model_data

    # word_embeddings の verbode をFalseに変更
    wym.we.verbose = False

    return wym


def make_wym_matcher_func_core(
    model_dir: str, data_columns: List[str], exclule_columns: List[str]
) -> Callable[[List[EntityPair], bool], np.array]:
    """Wym用のマッチャー関数を作成する"""
    df_empty = pd.DataFrame(columns=data_columns)
    # モデルの読み込み
    wym = Wym(
        df=df_empty,
        exclude_attrs=exclule_columns,
        model_files_path=model_dir,
        batch_size=BATCH_SIZE,
        reset_networks=False,
        verbose=False,
    )
    wym = modify_wym_member(wym)

    def proba_func(
        entity_pairs: List[EntityPair], expand_axis: bool = True
    ) -> np.array:
        # EntityPairをWYMのDataFrameに変換
        df = convert_entity_pairs_to_wym(entity_pairs)

        # 空のデータしかない場合、エラーとなるので、空のデータしかない場合は0を返す
        is_empty = df[wym.columns_to_use].replace("", np.nan).isna().all().all()
        if is_empty:
            return np.array([0] * len(entity_pairs))

        # マッチング確率の計算
        scores = wym.predict(df[wym.columns_to_use].copy(), return_data=False)

        # スコアを規格化 0.0 - 1.0 を -1.0 - 1.0 にする
        scores = 2 * scores - 1.0
        if expand_axis:
            # limeでは、1データに複数のラベルの結果がある場合が想定されているため、一軸増増やしたデータを作成
            return scores[:, np.newaxis]
        return scores

    return proba_func


def make_wym_matcher_func(
    dataset_name: str, model_root_dir: str, dataset_root_dir: str
) -> Callable[[List[EntityPair], bool], np.array]:
    dataset: SplittedDataset = load_dataset(dataset_name, dataset_root_dir)
    df_wfm = convert_dataset_to_wym(
        dataset.test.records.a,
        dataset.test.records.b,
        dataset.test.record_id_pairs,
        dataset.test.labels,
    )
    data_columns = df_wfm.columns.tolist().copy()
    exclude_columns = ["id", "left_id", "right_id", "label"]
    model_dir = os.path.join(model_root_dir, dataset_name)
    return make_wym_matcher_func_core(model_dir, data_columns, exclude_columns)


def test_proba_fn():
    target_dataset_name = dataset_names[TARGET_DATASET_ID]
    proba_fn = make_wym_matcher_func(
        target_dataset_name, model_root_dir, dataset_root_dir
    )

    dataset = load_dataset(target_dataset_name, dataset_root_dir)
    entity_pairs = []
    for idx in range(5):
        pair_id = dataset.test.record_id_pairs.iloc[idx : idx + 1]
        entity_l = Entity.from_dataframe(
            dataset.test.records.a[
                pair_id.iloc[0].loc["a.rid"] : pair_id.iloc[0]["a.rid"] + 1
            ]
        )
        entity_r = Entity.from_dataframe(
            dataset.test.records.b[
                pair_id.iloc[0].loc["b.rid"] : pair_id.iloc[0]["b.rid"] + 1
            ]
        )
        entity_pairs.append(EntityPair(entity_l, entity_r))
    entity_l = entity_pairs[0].entity_l.make_entity_by_deleting_segments(
        range(entity_pairs[0].entity_l.segment_size())
    )
    entity_r = entity_pairs[0].entity_r.make_entity_by_deleting_segments(
        range(entity_pairs[0].entity_r.segment_size())
    )
    empty_entity_pair = EntityPair(entity_l, entity_r)
    entity_pairs.append(empty_entity_pair)

    scores = proba_fn(entity_pairs, False)

    for pair, score in zip(entity_pairs, scores):
        display(pair.entity_l.to_dataframe())
        display(pair.entity_r.to_dataframe())
        print(score)
    empty_scores = proba_fn([empty_entity_pair], False)
    display(empty_entity_pair.to_dataframe())
    display(empty_entity_pair.to_dataframe())
    print(empty_scores)

    scores_single = [proba_fn([pair], False)[0] for pair in entity_pairs]

    np.testing.assert_array_almost_equal(scores, scores_single, decimal=5)


test_proba_fn()

## WYM 結果を読み込む、LimePair結果にコンバートする

In [ ]:
from typing import Dict, List, Tuple
import pathlib
import pickle
from dataclasses import dataclass
from pine.explainer import AttributionScore
from pine.entity import Entity, EntityPair
from pine.entity import MergedSegment


@dataclass
class WymResult:
    attributions: List[AttributionScore]
    match_score: float
    wym_intercept: float
    wym_pred_score: float
    wym_match_score: float
    dummy: float
    tokens_pairs: List[Tuple[str, str]]
    entity_pair: EntityPair


@dataclass
class LimeResultPair:
    attributions: List[AttributionScore]
    match_score: float
    lime_intercept: float
    lime_pred_score: float
    lime_match_score: float


def get_token_to_segment_id(entity: Entity):
    """エンティティ内のtokenに対応するセグメントのIDの辞書を返す"""
    token_to_segidx = {}
    for idx in range(entity.segment_size()):
        token = entity.get_segment_label(idx)
        token_to_segidx[token] = idx
    return token_to_segidx


def check_token_in_segments(token_to_segidx: Dict[str, int], tokens: List[str]):
    for token in tokens:
        if token not in token_to_segidx:
            print(
                f'WARNING: token "{token}" is not found in segments. {list(token_to_segidx.keys())}'
            )
    return


def convert_wym_to_limepair(wym_result: WymResult, topk:int) -> Tuple[LimeResultPair, EntityPair]:
    """Wym結果をLimeペア結果とEntityPairに変換する。
    EntityPairはExptanationに使われたtokenpairがマージされている状態のものを作る。
    attributionsもこれに合わせてインデックスを作成する。
    ただし、スコアの大きい順にtopk個のみを返す。"""

    # token_pairとentity_pairのインデックスの対応をとる
    entity_pair = wym_result.entity_pair
    token_l_to_idx_in_entity = get_token_to_segment_id(entity_pair.entity_l)
    token_r_to_idx_in_entity = get_token_to_segment_id(entity_pair.entity_r)
    # トークンが存在することを確かめる（ただし、ペア相手がいないことを示す"[UNP]"以外）
    check_token_in_segments(
        token_l_to_idx_in_entity, [t[0] for t in wym_result.tokens_pairs if t[0] != "[UNP]"]
    )
    check_token_in_segments(
        token_r_to_idx_in_entity, [t[1] for t in wym_result.tokens_pairs if t[1] != "[UNP]"]
    )

    # topk個のみのtoken_pairsを取り出す
    attributions_topk = sorted(
        wym_result.attributions, key=lambda attr: abs(attr.score), reverse=True
    )[:topk]
    token_pairs_topk = []
    for attr in attributions_topk:
        token_pairs_topk.append(wym_result.tokens_pairs[attr.index])

    # mergeしたentity_pairを作成
    merging_segment_list: List[MergedSegment] = []
    token_idx_pairs_topk = []
    for token_l, token_r in token_pairs_topk:
        idx_l = token_l_to_idx_in_entity.get(token_l)
        idx_r = token_r_to_idx_in_entity.get(token_r)
        token_idx_pairs_topk.append((idx_l, idx_r))
        # 見つからない原因が"[UNP]"ではない場合はスキップ
        if idx_l is None and token_l != "[UNP]":
            print(
                f"WARNING: token pair {token_l, token_r} = {idx_l, idx_r} is not found in entity pair."
            )
            token_idx_pairs_topk.append(())
            continue
        if idx_r is None and token_r != "[UNP]":
            print(
                f"WARNING: token pair {token_l, token_r} = {idx_l, idx_r} is not found in entity pair."
            )
            token_idx_pairs_topk.append(())
            continue
        if idx_l is None and idx_r is None:
            print(
                f"WARNING: token pair {token_l, token_r} = {idx_l, idx_r} is not found in entity pair."
            )
            token_idx_pairs_topk.append(())
            continue
        merge_seg = MergedSegment([], [])
        if idx_l is not None:
            merge_seg.segment_list_in_l.append(idx_l)
        if idx_r is not None:
            merge_seg.segment_list_in_r.append(idx_r)
        merging_segment_list.append(merge_seg)
    entity_pair_merged = entity_pair.make_entity_pair_by_merging_segment_list_only(
        merging_segment_list
    )

    # attributionsのインデックスを更新
    attributions_new = []
    for attr, (idx_l, idx_r) in zip(attributions_topk, token_idx_pairs_topk):
        idx_ls = [idx_l] if idx_l is not None else []
        idx_rs = [idx_r] if idx_r is not None else []
        idx_in_entity_pair = None
        for idx in range(entity_pair_merged.segment_size()):
            idx_ls_in_ep, idx_rs_in_ep = (
                entity_pair_merged.convert_segment_idx_to_entity_idx(idx)
            )
            if idx_ls_in_ep == idx_ls and idx_rs_in_ep == idx_rs:
                idx_in_entity_pair = idx
                break
        if idx_in_entity_pair is None:
            print(
                f"WARNING: token idx pair {idx_l, idx_r} is not found in entity pair."
            )
            continue
        attributions_new.append(AttributionScore(idx_in_entity_pair, attr.score))

    lime_pair_result = LimeResultPair(
        attributions_new,
        wym_result.match_score * 2 - 1.0,
        wym_result.wym_intercept,
        wym_result.wym_pred_score,
        wym_result.wym_match_score,
    )
    return lime_pair_result, entity_pair_merged


def load_wym_pair_explanation(
    target_dataset_name: str,
    pair_explanation_root_dir_path: pathlib.Path = None,
) -> Dict[int, Tuple[LimeResultPair, EntityPair]]:
    pair_explanations_dir_path = pair_explanation_root_dir_path / target_dataset_name
    results = {}
    for file_path in pair_explanations_dir_path.glob("*.pickle"):
        with file_path.open("rb") as f:
            data = pickle.load(f)
        # Wym結果を分かりやすいオブジェクトでラップする
        data_coverted = {}
        for id, result in data.items():
            data_coverted[id] = convert_wym_to_limepair(WymResult(*result), TOP_N)
        results.update(data_coverted)

    return results


def test_convert_wym_to_limepair():
    target_dataset_name = dataset_names[TARGET_DATASET_ID]
    pair_explanations_dir_path = pathlib.Path(wym_result_root_dir) / target_dataset_name
    wym_results = []
    for file_path in pair_explanations_dir_path.glob("*.pickle"):
        with file_path.open("rb") as f:
            data = pickle.load(f)
        for result in data.values():
            wym_results.append(WymResult(*result))
            if len(wym_results) >= 100:
                break
    for wym_result in wym_results:
        lime_result_pair, entity_pair = convert_wym_to_limepair(wym_result, TOP_N)
        # matcher_score は -1.0 から 1.0 になる
        assert wym_result.match_score * 2 - 1.0 == lime_result_pair.match_score
        # attributionsのインデックスに対応するtokenとスコアが同じであることを確認
        token_pair_scores_in_ep = []
        for attr in lime_result_pair.attributions:
            score = attr.score
            idx = attr.index
            idx_ls, idx_rs = entity_pair.convert_segment_idx_to_entity_idx(idx)
            tolen_l = (
                entity_pair.entity_l.get_segment_label(idx_ls[0])
                if len(idx_ls) > 0
                else "[UNP]"
            )
            tolen_r = (
                entity_pair.entity_r.get_segment_label(idx_rs[0])
                if len(idx_rs) > 0
                else "[UNP]"
            )
            token_pair_scores_in_ep.append(((tolen_l, tolen_r), score))
        token_pair_scores_in_wym = []
        for attr in sorted(
            wym_result.attributions, key=lambda attr: abs(attr.score), reverse=True
        ):
            token_pair_scores_in_wym.append((wym_result.tokens_pairs[attr.index], attr.score))
        for token_pair_score_in_ep, token_pair_score_in_wym in zip(token_pair_scores_in_ep, token_pair_scores_in_wym):
            assert token_pair_score_in_ep == token_pair_score_in_wym


def test_load_wym_pair_explanation():
    target_dataset_name = dataset_names[TARGET_DATASET_ID]
    ret = load_wym_pair_explanation(
        target_dataset_name,
        pathlib.Path(wym_result_root_dir),
    )
    for i, (pid, exp) in enumerate(ret.items()):
        print(pid)
        print(exp[0])
        print(exp[1].merged_segment_list)
        display(exp[1].entity_l.to_dataframe())
        display(exp[1].entity_r.to_dataframe())
        if i >= 10:
            break


test_convert_wym_to_limepair()
test_load_wym_pair_explanation()

### 難しいデータを抽出し、何件か出力

- 難しいデータ
  - label = match -> ペアの単語ベースのcos類似度が低い
  - label = unmatch -> ペアの単語ベースのcos類似度が高い 

In [ ]:
from collections import Counter
from math import sqrt
from typing import List

import pandas as pd
import lemon


def cosine_similarity(tokens_a: List[str], tokens_b: List[str]) -> float:
    """テキストのコサイン類似度を計算する"""
    # 各トークン集合の頻度をカウント
    counter_a = Counter(tokens_a)
    counter_b = Counter(tokens_b)

    # 全てのユニークなトークンを取得
    all_tokens = set(counter_a.keys()).union(set(counter_b.keys()))

    # トークンの頻度を基にベクトルを作成
    vec_a = [counter_a.get(token, 0) for token in all_tokens]
    vec_b = [counter_b.get(token, 0) for token in all_tokens]

    # ベクトルのドット積とマグニチュードを計算
    dot_product = sum([a * b for a, b in zip(vec_a, vec_b)])
    magnitude_a = sqrt(sum([a * a for a in vec_a]))
    magnitude_b = sqrt(sum([b * b for b in vec_b]))

    # コサイン類似度を計算
    if magnitude_a * magnitude_b == 0:
        return 0  # 0除算を避ける
    else:
        return dot_product / (magnitude_a * magnitude_b)


def extract_difficult_data(
    dataset: lemon.utils.datasets.Dataset, n: int = 10, pids: List = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """レコードペアのうち、難しいデータを、n件抽出する。

    label=1のとき
    - cosine類似度が小さいもの
    label=0のとき
    - cosine類似度が大きいもの

    return:
    - match 用の難しいデータ pidリスト
    - Unmatch 用の難しいデータ pidリスト
    """
    a_id_to_text = {}
    b_id_to_text = {}
    scores = []
    record_id_pairs_score = dataset.record_id_pairs.copy()
    if pids is not None:
        record_id_pairs_score = record_id_pairs_score.loc[pids]
    for pid, a_id, b_id in dataset.record_id_pairs.itertuples():
        texts_a = a_id_to_text.get(a_id, None)
        texts_b = b_id_to_text.get(b_id, None)
        if texts_a is None:
            entity_a = Entity.from_dataframe(dataset.records.a[a_id : a_id + 1])
            texts_a = [
                entity_a.get_segment_label(idx)
                for idx in range(entity_a.segment_size())
            ]
            a_id_to_text[a_id] = texts_a
        if texts_b is None:
            entity_b = Entity.from_dataframe(dataset.records.b[b_id : b_id + 1])
            texts_b = [
                entity_b.get_segment_label(idx)
                for idx in range(entity_b.segment_size())
            ]
            b_id_to_text[b_id] = texts_b
        cos_sim = cosine_similarity(texts_a, texts_b)
        scores.append(cos_sim)
    record_id_pairs_score["score"] = scores

    # label = True : cos_sim の小さい順 top n
    dataset_pair_match = record_id_pairs_score[dataset.labels == True].sort_values(
        "score", ascending=True
    )[:n]
    # label = false : cos_sim の大きい順 top n
    dataset_pair_unmatch = record_id_pairs_score[dataset.labels == False].sort_values(
        "score", ascending=False
    )[:n]

    return dataset_pair_match, dataset_pair_unmatch


def test_extract_difficult_data():
    target_dataset_name = dataset_names[TARGET_DATASET_ID]
    dataset = load_dataset(target_dataset_name, dataset_root_dir)
    match_pair, unmatch_pair = extract_difficult_data(dataset.test, 2)
    display(match_pair)
    for a_id, b_id, _ in match_pair.itertuples(index=False):
        display(dataset.test.records.a.loc[[a_id]])
        display(dataset.test.records.b.loc[[b_id]])
    display(unmatch_pair)
    for a_id, b_id, _ in unmatch_pair.itertuples(index=False):
        display(dataset.test.records.a.loc[[a_id]])
        display(dataset.test.records.b.loc[[b_id]])


test_extract_difficult_data()

In [ ]:
from contextlib import contextmanager

from pine.explainer import AttributionScore


@dataclass
class PairSegment:
    index_l: int
    index_r: int
    score: float
    match_score_diff: float
    del_left: bool


@contextmanager
def option_context(*args, **kwargs):
    original_options = {opt: pd.get_option(opt) for opt in kwargs.keys()}
    pd.set_option(*args, **kwargs)
    try:
        yield
    finally:
        for opt, value in original_options.items():
            pd.set_option(opt, value)


def display_pair_tokens(
    entity_l: Entity,
    entity_r: Entity,
    pair_segments: List[PairSegment],
    topk: int = 5,
):
    for pair_seg in pair_segments[:topk]:
        tokens_l = (
            entity_l.get_segment_label(pair_seg.index_l)
            if pair_seg.index_l is not None
            else None
        )
        tokens_r = (
            entity_r.get_segment_label(pair_seg.index_r)
            if pair_seg.index_r is not None
            else None
        )
        print(
            tokens_l,
            tokens_r,
            pair_seg.score,
            pair_seg.match_score_diff,
            "del_left" if pair_seg.del_left else "del_right",
        )


def display_pair_explanation(
    pair_explanation: LimeResultPair,
    entity_pair: EntityPair,
):
    for attr_pair in pair_explanation.attributions:
        token_lr = entity_pair.get_segment_label(attr_pair.index)
        print(
            token_lr,
            attr_pair.score,
        )


def display_result(
    entity_l: Entity,
    entity_r: Entity,
    label: int,
    pair_segment: List[PairSegment],
    lime_result: LimeResultPair,
    topk: int,
    pair_explanation: LimeResultPair,
    pair_explanation_entity_pair: EntityPair,
):
    print("===============================================================")
    with option_context("display.max_colwidth", 200):
        display(entity_l.to_dataframe())
        display(entity_r.to_dataframe())
    print("LABEL =", label)
    print(
        "Match Score =",
        lime_result.match_score
        if lime_result is not None
        else pair_explanation.match_score,
    )
    print("======= Related pair tokens")
    display_pair_tokens(entity_l, entity_r, pair_segment, topk)
    display_pair_explanation(pair_explanation, pair_explanation_entity_pair)

In [ ]:
import itertools


def display_difficult_data(dataset_name: str, matcher_name: str, topk: int = 10):
    dataset = load_dataset(dataset_name, dataset_root_dir)
    pair_explanations = load_wym_pair_explanation(
        dataset_name, pathlib.Path(wym_result_root_dir)
    )
    match_pair_df, unmatch_pair_df = extract_difficult_data(dataset.test, 10)
    print("dataset_name:", dataset_name)
    print("matcher_name:", matcher_name)
    for pid, l_id, r_id, _ in itertools.chain(
        match_pair_df.itertuples(), unmatch_pair_df.itertuples()
    ):
        entity_l = Entity.from_dataframe(dataset.test.records.a.loc[[l_id]])
        entity_r = Entity.from_dataframe(dataset.test.records.b.loc[[r_id]])
        label = dataset.test.labels.loc[pid]
        pair_explanation, pair_explanation_entity_pair = pair_explanations[pid]
        pair_segs = []
        for merge_seg in pair_explanation_entity_pair.merged_segment_list:
            pair_segs.append(
                PairSegment(
                    merge_seg.segment_list_in_l[0]
                    if len(merge_seg.segment_list_in_l) > 0
                    else None,
                    merge_seg.segment_list_in_r[0]
                    if len(merge_seg.segment_list_in_r) > 0
                    else None,
                    0,
                    0,
                    False,
                )
            )
        display_result(
            entity_l,
            entity_r,
            label,
            pair_segs,
            None,
            10,
            pair_explanation,
            pair_explanation_entity_pair,
        )


display_difficult_data(dataset_names[TARGET_DATASET_ID], matcher_names[0], TOP_N)

### 評価用データ作成

- faithful用
  - dataid -> k -> match_score(k token pair まで追加した時の match score)
- Contributuon用
  - dataid -> k -> match_score(k token pair まで削除した時の match score)
- CosSim用
  - dataid -> k -> cos_sim(k token pair 番目のcos sim)
- ペアで動作しているか
  - dataid -> k 
    - k ペア削除した時のMatchScore
    - kペアの右側を削除した時のMatchScore
    - kペアの左側を削除した時のMatchScore

## failthful用

In [ ]:
from pine.entity import Attribute


def make_match_scores_with_token_pairs(
    entity_pair: EntityPair,
    match_func,
    pair_segments: List[PairSegment],
):
    """pair_segmentsのペアのみを残した場合のスコアリストを返す。
    tokenを残す位置は、元のentity_pairの位置と同じ位置とする。

    pair_segmentsが5個の場合、6個の以下のスコアリストを返す。
    scores[0] = 何も残さない時のスコア
    scores[1] = pair_segments[0]のみ残した場合のスコア
    scores[2] = pair_segments[0]とpair_segments[1]を残した場合のスコア
    scores[3] = pair_segments[0]からpair_segments[2]を残した場合のスコア
    scores[4] = pair_segments[0]からpair_segments[3]を残した場合のスコア
    scores[5] = pair_segments[0]からpair_segments[4]を残した場合のスコア
    """
    entity_pairs = []
    # 何も残さない
    delete_segments = list(range(entity_pair.segment_size()))
    entity_pair_0 = entity_pair.make_entity_pair_by_deleting_segments(delete_segments)
    entity_pairs.append(entity_pair_0)

    for i in range(len(pair_segments)):
        remain_segments_l = [
            pair_token.index_l
            for pair_token in pair_segments[: i + 1]
            if pair_token.index_l is not None
        ]
        delete_segments_l = [
            idx
            for idx in range(entity_pair.entity_l.segment_size())
            if idx not in remain_segments_l
        ]
        entity_l = entity_pair.entity_l.make_entity_by_deleting_segments(
            delete_segments_l
        )

        remain_segments_r = [
            pair_token.index_r
            for pair_token in pair_segments[: i + 1]
            if pair_token.index_r is not None
        ]
        delete_segments_r = [
            idx
            for idx in range(entity_pair.entity_r.segment_size())
            if idx not in remain_segments_r
        ]
        entity_r = entity_pair.entity_r.make_entity_by_deleting_segments(
            delete_segments_r
        )
        entity_pairs.append(EntityPair(entity_l, entity_r))

    scores = match_func(entity_pairs, False)

    # scores = []
    # for entity_pair_ in entity_pairs:
    #     score = match_func([entity_pair_], False)
    #     scores.append(score[0])
    # scores = np.array(scores, dtype=np.float16)

    return scores


def test_make_match_scores_with_token_pairs():
    target_dataset_name = "structured_amazon_google"
    attr_list_1 = [
        Attribute("title", "apple iphone 13", "string"),
        Attribute("manufacturer", "apple", "string"),
        Attribute("price", 10, "Float64"),
    ]
    attr_list_2 = [
        Attribute("title", "apple iphone 12 pro", "string"),
        Attribute("manufacturer", "apple", "string"),
        Attribute("price", 10, "Float64"),
    ]
    entity_pair = EntityPair(Entity(attr_list_1), Entity(attr_list_2))
    func = make_wym_matcher_func(target_dataset_name, model_root_dir, dataset_root_dir)
    scores = make_match_scores_with_token_pairs(
        entity_pair,
        func,
        [
            PairSegment(0, 0, 0, 0, True),
            PairSegment(0, 1, 0, 0, True),
            PairSegment(1, 1, 0, 0, True),
            PairSegment(3, None, 0, 0, True),
        ],
    )

    entity_pairs_expected = [
        EntityPair(
            Entity(attr_list_1).make_entity_by_deleting_segments([0, 1, 2, 3]),
            Entity(attr_list_2).make_entity_by_deleting_segments([0, 1, 2, 3, 4]),
        ),
        EntityPair(
            Entity(attr_list_1).make_entity_by_deleting_segments([1, 2, 3]),
            Entity(attr_list_2).make_entity_by_deleting_segments([1, 2, 3, 4]),
        ),
        EntityPair(
            Entity(attr_list_1).make_entity_by_deleting_segments([1, 2, 3]),
            Entity(attr_list_2).make_entity_by_deleting_segments([2, 3, 4]),
        ),
        EntityPair(
            Entity(attr_list_1).make_entity_by_deleting_segments([2, 3]),
            Entity(attr_list_2).make_entity_by_deleting_segments([2, 3, 4]),
        ),
        EntityPair(
            Entity(attr_list_1).make_entity_by_deleting_segments([2]),
            Entity(attr_list_2).make_entity_by_deleting_segments([2, 3, 4]),
        ),
    ]
    scores_expected = [func([entity_pair_expected], False)[0] for entity_pair_expected in entity_pairs_expected]


    df_singles = [convert_entity_pairs_to_wym([entity_pair_]) for entity_pair_ in entity_pairs_expected]
    df_batched = convert_entity_pairs_to_wym(entity_pairs_expected)

    print("SINGLE")
    for df_single in df_singles:
        display(df_single)
        print(df_single.dtypes)
        assert df_single.columns.tolist() == df_batched.columns.tolist()
    print("BATCH")
    display(df_batched)
    print(df_batched.dtypes)

    data_cocolumns = df_singles[0].columns.to_list()
    exclude_columns = ["id", "left_id", "right_id"]
    for col in exclude_columns:
        data_cocolumns.remove(col)

    df_singles = pd.concat(df_singles)
    for col in data_cocolumns:
        pd.testing.assert_series_equal(df_singles[col].reset_index(drop=True), df_batched[col].reset_index(drop=True))

    np.testing.assert_almost_equal(scores, scores_expected, decimal=5)


test_make_match_scores_with_token_pairs()


In [ ]:
from tqdm import tqdm


def eval_match_score_remain(
    entity_pairs: Dict[int, EntityPair],
    matcher_func: Callable,
    results: Dict[int, List[PairSegment]],
    out_dir_path: pathlib.Path,
):
    score_remains = {}
    score_remains_file_path = out_dir_path / "match_score_remains.pickle"
    if score_remains_file_path.exists():
        # すでに結果があればロードする
        with score_remains_file_path.open("rb") as f_in:
            score_remains = pickle.load(f_in)
    else:
        # 結果ファイルがなければ作成する
        for pid, entity_pair in tqdm(entity_pairs.items()):
            segment_pairs = results[pid]
            score_remains[pid] = make_match_scores_with_token_pairs(
                entity_pair, matcher_func, segment_pairs
            )

        with score_remains_file_path.open("wb") as f_out:
            pickle.dump(score_remains, f_out)
    return score_remains


In [ ]:
import matplotlib.pyplot as plt


def _calc_post_hoc_acc(
    rank,
    match_scores_pertub,
    lime_pair_results: Dict[int, LimeResultPair],
):
    match_scores_pertub_arr = np.empty((len(match_scores_pertub),), dtype=np.float)
    match_scores_org_arr = np.empty((len(match_scores_pertub),), dtype=np.float)
    for i, pid in enumerate(match_scores_pertub):
        match_scores_pertub_ = match_scores_pertub[pid]
        # 削除できるものがない場合、この前のスコアを採用
        if match_scores_pertub_.size < rank + 1:
            match_scores_pertub_ = np.pad(
                match_scores_pertub[pid],
                (0, rank + 1 - match_scores_pertub_.size),
                "edge",
            )
        # print(match_scores_pertub[pid])
        # print(match_scores_pertub_)
        match_scores_pertub_arr[i] = match_scores_pertub_[rank]
        match_scores_org_arr[i] = lime_pair_results[pid].match_score
    valid_indices = ~np.isnan(match_scores_pertub_arr)
    y = (match_scores_org_arr[valid_indices] > 0).astype(int)
    y_pred = (match_scores_pertub_arr[valid_indices] > 0).astype(int)
    print("len=", len(y_pred), "co=", (y == y_pred).sum())
    return np.mean(y == y_pred)


def display_eval_match_score_remain(target_dataset_name, top_n):
    for target_matcher_name in matcher_names:
        if target_matcher_name == "wym":
            matcher_func = make_wym_matcher_func(
                target_dataset_name, model_root_dir, dataset_root_dir
            )
        else:
            ValueError(f"Not supported matcher name.{target_matcher_name}")
        pair_explanations = load_wym_pair_explanation(
            target_dataset_name,
            pathlib.Path(wym_result_root_dir),
        )
        pair_segs_all = {}
        entity_pairs_all = {}
        pair_exp_all = {}
        for pid, (pair_ex, pair_ex_entity_pair) in pair_explanations.items():
            pair_segs = []
            # attribution_scoreの絶対値が大きい順にペアを作成
            for attr in sorted(
                pair_ex.attributions, key=lambda x: abs(x.score), reverse=True
            ):
                if len(pair_segs) >= top_n:
                    break
                segment_list_in_l = pair_ex_entity_pair.merged_segment_list[
                    attr.index
                ].segment_list_in_l
                segment_list_in_r = pair_ex_entity_pair.merged_segment_list[
                    attr.index
                ].segment_list_in_r
                l_idx = segment_list_in_l[0] if len(segment_list_in_l) > 0 else None
                r_idx = segment_list_in_r[0] if len(segment_list_in_r) > 0 else None
                pair_segs.append(PairSegment(l_idx, r_idx, attr.score, 0, True))
            pair_segs_all[pid] = pair_segs
            entity_pairs_all[pid] = EntityPair(
                pair_ex_entity_pair.entity_l, pair_ex_entity_pair.entity_r
            )
            pair_exp_all[pid] = pair_ex

        out_dir_path = (
            pathlib.Path(out_root_dir) / target_matcher_name / target_dataset_name
        )
        out_dir_path.mkdir(parents=True, exist_ok=True)

        score_remains = eval_match_score_remain(
            entity_pairs_all, matcher_func, pair_segs_all, out_dir_path
        )

        post_hoc_accs = []
        for i in range(top_n + 1):
            acc = _calc_post_hoc_acc(i, score_remains, pair_exp_all)
            post_hoc_accs.append(acc)

        print(post_hoc_accs)
        plt.figure(figsize=(15, 6))
        plt.subplot(1, 2, 1)  # 1行2列のグリッドの最初のプロット
        plt.plot(post_hoc_accs, marker="o")
        plt.xlabel("token pairs")
        plt.ylabel("accuracy")
        plt.title("post-hoc accuracy")
        plt.grid(True)
        plt.tight_layout()  # グラフ同士の間隔を調整
        plt.show()


display_eval_match_score_remain(dataset_names[TARGET_DATASET_ID], TOP_N)

## Contribution

In [ ]:
def make_match_scores_with_token_pairs_deleted(
    entity_pair: EntityPair,
    match_func,
    pair_segments: List[PairSegment],
):
    """pair_segmentsのペアを削除した場合のスコアリストを返す。

    pair_segmentsが5個の場合、6個の以下のスコアリストを返す。
    scores[0] = 何も残さない時のスコア
    scores[1] = pair_segments[0]のみ削除した場合のスコア
    scores[2] = pair_segments[0]とpair_segments[1]を削除した場合のスコア
    scores[3] = pair_segments[0]からpair_segments[2]を削除した場合のスコア
    scores[4] = pair_segments[0]からpair_segments[3]を削除した場合のスコア
    scores[5] = pair_segments[0]からpair_segments[4]を削除した場合のスコア
    """
    entity_pairs = []
    # そのまま
    entity_pairs.append(entity_pair)

    for i in range(len(pair_segments)):
        delete_segments_l = [
            pair_token.index_l
            for pair_token in pair_segments[: i + 1]
            if pair_token.index_l is not None
        ]
        entity_l = entity_pair.entity_l.make_entity_by_deleting_segments(
            delete_segments_l
        )
        delete_segments_r = [
            pair_token.index_r
            for pair_token in pair_segments[: i + 1]
            if pair_token.index_r is not None
        ]
        entity_r = entity_pair.entity_r.make_entity_by_deleting_segments(
            delete_segments_r
        )
        entity_pairs.append(EntityPair(entity_l, entity_r))

    scores = match_func(entity_pairs, False)

    # for i, entity_pair_ in enumerate(entity_pairs):
    #     print(*entity_pair_.to_dataframe())
    #     print(scores[i])

    return scores


def test_make_match_scores_with_token_pairs_deleted():
    target_dataset_name = "structured_amazon_google"
    attr_list_1 = [
        Attribute("title", "apple iphone 13", "string"),
        Attribute("manufacturer", "apple", "string"),
        Attribute("price", 10, "Float64"),
    ]
    attr_list_2 = [
        Attribute("title", "apple iphone 12", "string"),
        Attribute("manufacturer", "apple", "string"),
        Attribute("price", 10, "Float64"),
    ]
    entity_pair = EntityPair(Entity(attr_list_1), Entity(attr_list_2))
    func = make_wym_matcher_func(target_dataset_name, model_root_dir, dataset_root_dir)
    scores = make_match_scores_with_token_pairs_deleted(
        entity_pair,
        func,
        [
            PairSegment(0, 0, 0, 0, True),
            PairSegment(0, 1, 0, 0, True),
            PairSegment(1, 1, 0, 0, True),
            PairSegment(3, None, 0, 0, True),
        ],
    )
    print(scores)

    entity_pairs_expected = [
        EntityPair(
            Entity(attr_list_1),
            Entity(attr_list_2),
        ),
        EntityPair(
            Entity(attr_list_1).make_entity_by_deleting_segments([0]),
            Entity(attr_list_2).make_entity_by_deleting_segments([0]),
        ),
        EntityPair(
            Entity(attr_list_1).make_entity_by_deleting_segments([0]),
            Entity(attr_list_2).make_entity_by_deleting_segments([0, 1]),
        ),
        EntityPair(
            Entity(attr_list_1).make_entity_by_deleting_segments([0, 1]),
            Entity(attr_list_2).make_entity_by_deleting_segments([0, 1]),
        ),
        EntityPair(
            Entity(attr_list_1).make_entity_by_deleting_segments([0, 1, 3]),
            Entity(attr_list_2).make_entity_by_deleting_segments([0, 1]),
        ),
    ]

    scores_expected = func(entity_pairs_expected, False)

    np.testing.assert_almost_equal(scores, scores_expected)


test_make_match_scores_with_token_pairs_deleted()

In [ ]:
def dump_match_score_delete(
    target_dataset_name: str, target_matcher_name: str, out_root_dir: pathlib.Path
):
    score_deletes = {}
    out_dir_path = (
        pathlib.Path(out_root_dir) / target_matcher_name / target_dataset_name
    )
    out_dir_path.mkdir(parents=True, exist_ok=True)
    score_deletes_file_path = out_dir_path / "match_score_deletes.pickle"
    if score_deletes_file_path.exists():
        # すでに結果があればロードする
        with score_deletes_file_path.open("rb") as f_in:
            score_deletes = pickle.load(f_in)
    else:
        # 結果ファイルがなければ作成する
        dataset = load_dataset(target_dataset_name, dataset_root_dir)
        if target_matcher_name == "wym":
            matcher_func = make_wym_matcher_func(
                target_dataset_name, model_root_dir, dataset_root_dir
            )
        else:
            raise ValueError(f"matcher_func is not supported. {target_matcher_name}")
        pair_explanations = load_wym_pair_explanation(
            target_dataset_name, pathlib.Path(wym_result_root_dir)
        )

        for pid in tqdm(
            pair_explanations.keys(),
            total=len(pair_explanations),
        ):
            segment_pairs = []
            pair_ex, pair_ex_entity_pair = pair_explanations[pid]
            for attr in sorted(
                pair_ex.attributions, key=lambda x: abs(x.score), reverse=True
            ):
                segment_list_in_l = pair_ex_entity_pair.merged_segment_list[
                    attr.index
                ].segment_list_in_l
                segment_list_in_r = pair_ex_entity_pair.merged_segment_list[
                    attr.index
                ].segment_list_in_r
                l_idx = segment_list_in_l[0] if len(segment_list_in_l) > 0 else None
                r_idx = segment_list_in_r[0] if len(segment_list_in_r) > 0 else None
                segment_pairs.append(PairSegment(l_idx, r_idx, attr.score, 0, True))

            if dataset.test.labels.loc[pid]:
                # match のentity pair の場合、match に有効なものから順番（attr.scoreの降順）に削除対象
                segment_pairs = sorted(
                    segment_pairs, key=lambda x: x.score, reverse=True
                )
            else:
                # unmatch のentity pair の場合、unmatch に有効なものから順番（attr.scoreの昇順）に削除対象
                segment_pairs = sorted(segment_pairs, key=lambda x: x.score)

            score_deletes[pid] = make_match_scores_with_token_pairs_deleted(
                pair_ex_entity_pair, matcher_func, segment_pairs
            )

        with score_deletes_file_path.open("wb") as f_out:
            pickle.dump(score_deletes, f_out)
    return score_deletes


def load_dump_match_score_delete(
    target_dataset_name: str, target_matcher_name: str, out_root_dir: pathlib.Path
):
    out_dir_path = (
        pathlib.Path(out_root_dir) / target_matcher_name / target_dataset_name
    )
    score_deletes_file_path = out_dir_path / "match_score_deletes.pickle"
    with score_deletes_file_path.open("rb") as f_in:
        score_deletes = pickle.load(f_in)
    return score_deletes


In [ ]:
def _calc_posthoc_f1(
    rank, match_scores_pertub: Dict[int, np.array], correct: pd.Series
):
    match_scores_pertub_arr = np.empty((len(match_scores_pertub),), dtype=np.float)
    y = correct.loc[match_scores_pertub.keys()].to_numpy()
    for i, pid in enumerate(match_scores_pertub):
        match_scores_pertub_ = match_scores_pertub[pid]
        # 削除できるものがない場合、この前のスコアを採用
        if match_scores_pertub_.size < rank + 1:
            match_scores_pertub_ = np.pad(
                match_scores_pertub[pid],
                (0, rank + 1 - match_scores_pertub_.size),
                "edge",
            )
        # print(match_scores_pertub[pid])
        # print(match_scores_pertub_)
        match_scores_pertub_arr[i] = match_scores_pertub_[rank]
    valid_indices = ~np.isnan(match_scores_pertub_arr)
    y = y[valid_indices].astype(int)
    y_pred = (match_scores_pertub_arr[valid_indices] > 0).astype(int)
    tp = np.sum((y == 1) & (y_pred == 1))
    fp = np.sum((y == 0) & (y_pred == 1))
    fn = np.sum((y == 1) & (y_pred == 0))
    print("len=", len(y), "tp=", tp, "fp=", fp, "fn=", fn)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (
        (2 * precision * recall) / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )
    return f1


def display_eval_match_score_delete(target_dataset_name, top_n):
    dataset = load_dataset(target_dataset_name, dataset_root_dir)
    for target_matcher_name in matcher_names:
        score_deletes = dump_match_score_delete(
            target_dataset_name, target_matcher_name, pathlib.Path(out_root_dir)
        )

        post_hoc_f1 = []
        for i in range(top_n + 1):
            f1 = _calc_posthoc_f1(i, score_deletes, dataset.test.labels)
            post_hoc_f1.append(f1)

        print(post_hoc_f1)
        plt.figure(figsize=(15, 6))
        plt.subplot(1, 2, 1)  # 1行2列のグリッドの最初のプロット
        plt.plot(post_hoc_f1, marker="o")
        plt.xlabel("token pairs")
        plt.ylabel("f1")
        plt.title("post-hoc f1")
        plt.grid(True)
        plt.tight_layout()  # グラフ同士の間隔を調整
        plt.show()


display_eval_match_score_delete(dataset_names[TARGET_DATASET_ID], TOP_N)

### ペアの単語が類似しているか計測

- BERTでペアの単語cosine類似度を計測
- ペア単語のcosine類似度を累積

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
from scipy.spatial.distance import cosine

device = "cuda" if torch.cuda.is_available() else "cpu"

# モデルとトークナイザーの準備
bert_model_name = "bert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
bert_model = AutoModel.from_pretrained(bert_model_name).to(device)

In [ ]:
def calculate_cosine_similarities_with_mean_pooling(word_pairs) -> np.array:
    # 0件なら0件で返す
    if len(word_pairs) == 0:
        return np.array([])

    word1_list = [pair[0] for pair in word_pairs]
    word2_list = [pair[1] for pair in word_pairs]

    inputs1 = bert_tokenizer(
        word1_list, return_tensors="pt", truncation=True, padding=True, max_length=128
    ).to(bert_model.device)
    inputs2 = bert_tokenizer(
        word2_list, return_tensors="pt", truncation=True, padding=True, max_length=128
    ).to(bert_model.device)

    with torch.no_grad():
        outputs1 = bert_model(**inputs1)
        outputs2 = bert_model(**inputs2)

    # 平均Poolingを行う
    mask1 = (
        inputs1["attention_mask"]
        .unsqueeze(-1)
        .expand_as(outputs1.last_hidden_state)
        .float()
    )
    mask2 = (
        inputs2["attention_mask"]
        .unsqueeze(-1)
        .expand_as(outputs2.last_hidden_state)
        .float()
    )

    embed1 = torch.sum(outputs1.last_hidden_state * mask1, 1) / mask1.sum(1)
    embed2 = torch.sum(outputs2.last_hidden_state * mask2, 1) / mask2.sum(1)

    # コサイン類似度を計算し、numpy配列として返す
    similarities = (
        torch.nn.functional.cosine_similarity(embed1, embed2).to("cpu").numpy()
    )
    return similarities


def calc_segment_pair_cos_sim_each(
    entity_pair: EntityPair,
    pair_segments: List[PairSegment],
) -> np.array:
    word_pairs = []
    no_pair_idx = []
    for idx, pair_seg in enumerate(pair_segments):
        if pair_seg.index_l is not None and pair_seg.index_r is not None:
            word_l = entity_pair.entity_l.get_segment_label(pair_seg.index_l)
            word_r = entity_pair.entity_r.get_segment_label(pair_seg.index_r)
            word_pairs.append((word_l, word_r))
        else:
            no_pair_idx.append(idx)

    scores = calculate_cosine_similarities_with_mean_pooling(word_pairs)
    # どちらかがnoneだった場合計算できないので、Noneを挿入する
    result = [None] * len(pair_segments)
    valid_idx = 0

    for i in range(len(pair_segments)):
        if i in no_pair_idx:
            result[i] = None
        else:
            result[i] = scores[valid_idx]
            valid_idx += 1

    return np.array(result)


def test_calculate_cosine_similarities_with_mean_pooling():
    # 類似度の計算
    word_pairs = [
        ("dog", "dog"),
        ("dog", "cat"),
        ("computer", "program"),
        ("book", "page"),
    ]
    similarities = calculate_cosine_similarities_with_mean_pooling(word_pairs)

    for (word1, word2), sim in zip(word_pairs, similarities):
        print(f"コサイン類似度 between '{word1}' and '{word2}': {sim}")


test_calculate_cosine_similarities_with_mean_pooling()

In [ ]:
def eval_pair_cosine(
    entity_pairs: Dict[int, EntityPair],
    results,
    out_dir_path: pathlib.Path,
):
    """各ペアのコサイン類似度をBERTで計測"""
    cosine_sims_each = {}
    cosine_sims_each_file_path = out_dir_path / "cosine_sims_each.pickle"
    if cosine_sims_each_file_path.exists():
        with cosine_sims_each_file_path.open("rb") as f_in:
            cosine_sims_each = pickle.load(f_in)
    else:
        # 途中結果ファイルがなければ作成する
        for pid in tqdm(
            entity_pairs.keys(),
            total=len(entity_pairs),
        ):
            result_data = results[pid]
            entity_pair = entity_pairs[pid]
            cosine_sims_each[pid] = calc_segment_pair_cos_sim_each(
                entity_pair, result_data
            )

        with cosine_sims_each_file_path.open("wb") as f_out:
            pickle.dump(cosine_sims_each, f_out)
    return cosine_sims_each

In [ ]:
def _get_average(
    scores: List[np.ndarray],
    max_length: int = None,
    mode: str = "edge",
    dump: bool = False,
) -> np.ndarray:
    if max_length is None:
        max_length = max([len(s) for s in scores])
    if mode == "constant":
        scores = [
            np.pad(
                s[:max_length].astype(float),
                (0, max_length - s[:max_length].shape[0]),
                mode="constant",
                constant_values=np.nan,
            )
            if len(s) != 0
            else np.array([np.nan] * max_length)
            for s in scores
        ]
    else:
        scores = [
            np.pad(
                s[:max_length].astype(float),
                (0, max_length - s[:max_length].shape[0]),
                mode=mode,
            )
            if len(s) != 0
            else np.array([np.nan] * max_length)
            for s in scores
        ]
    if dump:
        print(scores)
    if len(scores) == 0:
        return np.zeros((max_length,))
    scores = np.vstack(scores)
    return np.nanmean(scores, axis=0)


def cumsum_with_none(arr):
    # None を 0 に置き換え
    arr = np.where(arr == None, 0, arr).astype(np.float64)
    # 累積和を計算
    return np.cumsum(arr)


def display_eval_pair_cosine(target_dataset_name: str, top_n: int):
    for target_matcher_name in matcher_names:
        pair_explanations = load_wym_pair_explanation(
            target_dataset_name, pathlib.Path(wym_result_root_dir)
        )
        entity_pairs = {}
        results = {}
        for pid, (pair_ex, pair_ex_entity_pair) in pair_explanations.items():
            pair_segs = []
            for attr in sorted(
                pair_ex.attributions, key=lambda x: abs(x.score), reverse=True
            ):
                segment_list_in_l = pair_ex_entity_pair.merged_segment_list[
                    attr.index
                ].segment_list_in_l
                segment_list_in_r = pair_ex_entity_pair.merged_segment_list[
                    attr.index
                ].segment_list_in_r
                l_idx = segment_list_in_l[0] if len(segment_list_in_l) > 0 else None
                r_idx = segment_list_in_r[0] if len(segment_list_in_r) > 0 else None
                pair_segs.append(PairSegment(l_idx, r_idx, attr.score, 0, True))
            results[pid] = pair_segs
            entity_pairs[pid] = EntityPair(
                pair_ex_entity_pair.entity_l, pair_ex_entity_pair.entity_r
            )

        out_dir_path = (
            pathlib.Path(out_root_dir) / target_matcher_name / target_dataset_name
        )
        out_dir_path.mkdir(parents=True, exist_ok=True)

        cos_sims_each = eval_pair_cosine(entity_pairs, results, out_dir_path)

        # match の entity pair と、unmatch の entity pairを分ける
        cos_sims_each_match = {}
        cos_sims_each_unmatch = {}
        for pid, (pair_ex, pair_ex_entity_pair) in pair_explanations.items():
            match_score_org = pair_ex.match_score
            if match_score_org > 0:
                cos_sims_each_match[pid] = cos_sims_each[pid]
            else:
                cos_sims_each_unmatch[pid] = cos_sims_each[pid]

        # コサイン値の積み上げの平均
        cos_sims_match_averaged = _get_average(
            [cumsum_with_none(arr) for arr in cos_sims_each_match.values()],
            max_length=top_n,
        )
        cos_sims_unmatch_averaged = _get_average(
            [cumsum_with_none(arr) for arr in cos_sims_each_unmatch.values()],
            max_length=top_n,
        )
        if len(cos_sims_match_averaged) <= top_n:
            no_data_length = top_n - len(cos_sims_match_averaged)
            cos_sims_match_averaged = np.pad(
                cos_sims_match_averaged,
                (0, no_data_length),
                mode="constant",
                constant_values=np.nan,
            )
        if len(cos_sims_unmatch_averaged) <= top_n:
            no_data_length = top_n - len(cos_sims_unmatch_averaged)
            cos_sims_unmatch_averaged = np.pad(
                cos_sims_unmatch_averaged,
                (0, no_data_length),
                mode="constant",
                constant_values=np.nan,
            )

        plt.figure(figsize=(15, 6))
        plt.subplot(1, 2, 1)  # 1行2列のグリッドの最初のプロット
        plt.plot(range(1, top_n + 1), cos_sims_match_averaged, marker="o")
        plt.xlim(1, 5)
        plt.xlabel("Pair tokens")
        plt.ylabel("cos sim")
        plt.title("Match Pair tokens")
        plt.grid(True)
        plt.subplot(1, 2, 2)  # 1行2列のグリッドの二つ目のプロット
        plt.plot(range(1, top_n + 1), cos_sims_unmatch_averaged, marker="o")
        plt.xlim(1, 5)
        plt.xlabel("Pair tokens")
        plt.ylabel("cos sim")
        plt.title("UnMatch Pair tokens")
        plt.grid(True)
        plt.tight_layout()  # グラフ同士の間隔を調整
        plt.show()
    return


display_eval_pair_cosine(dataset_names[TARGET_DATASET_ID], TOP_N)

### 考察用データ出力
- Entity_L, Entity_R, is_match, match_scoretoken_L_1, org_score, token_R_1, org_score, token_L_2, org_score, token_R_2, org_score, token_L_3, org_score, token_R_3, org_score, token_L_4, org_score, token_R_4, org_score, token_L_5, org_score, token_R_5, org_score

In [ ]:
import pandas as pd
from typing import Any


def record_to_prefixed_dict(df, index, prefix):
    """
    指定された index の DataFrame のレコードを辞書に変換し、
    カラム名に指定された接頭辞を追加します。

    Parameters:
    - df: pd.DataFrame, 変換する DataFrame
    - index: int, 変換するレコードの index
    - prefix: str, カラム名に追加する接頭辞

    Returns:
    - dict: 指定されたレコードの辞書表現。キーは接頭辞を追加したカラム名です。
    """
    # 指定された index のレコードを取得
    record = df.loc[index]

    # カラム名に接頭辞を追加して辞書を作成
    prefixed_dict = {f"{prefix}_{key}": value for key, value in record.items()}

    return prefixed_dict


def make_dump_data(
    dataset: lemon.utils.datasets.SplittedDataset,
    pair_explanations: Dict[int, Tuple[LimeResultPair, EntityPair]],
    top_n: int,
    score_deletes: Dict[int, np.ndarray],
):
    """ダンプデータを作成する"""
    dump_data = []
    for pid, l_id, r_id in tqdm(
        dataset.test.record_id_pairs.itertuples(),
        total=len(dataset.test.record_id_pairs),
    ):
        # lime_result_org = lime_result[pid][(None, None)]
        lime_result_pair, entity_pair_on_pair_explanation = pair_explanations[pid]
        score_delete = score_deletes[pid]

        # entity_l = Entity.from_dataframe(dataset.test.records.a.loc[[l_id]])
        # entity_r = Entity.from_dataframe(dataset.test.records.b.loc[[r_id]])
        entity_l = entity_pair_on_pair_explanation.entity_l
        entity_r = entity_pair_on_pair_explanation.entity_r

        data = {}
        data["Pair_ID"] = pid
        data.update(record_to_prefixed_dict(dataset.test.records.a, l_id, "Entity_L"))
        data.update(record_to_prefixed_dict(dataset.test.records.b, r_id, "Entity_R"))
        data["is_match"] = dataset.test.labels.loc[pid]
        data["match_score"] = lime_result_pair.match_score

        # attributions_l_org = {x.index: x.score for x in lime_result_org.attributions_l}
        # attributions_r_org = {x.index: x.score for x in lime_result_org.attributions_r}
        for i, attr in enumerate(
            sorted(
                lime_result_pair.attributions,
                key=lambda attr: abs(attr.score),
                reverse=True,
            )[:top_n]
        ):
            segment_list_in_l = entity_pair_on_pair_explanation.merged_segment_list[
                attr.index
            ].segment_list_in_l
            segment_list_in_r = entity_pair_on_pair_explanation.merged_segment_list[
                attr.index
            ].segment_list_in_r
            index_l = segment_list_in_l[0] if len(segment_list_in_l) > 0 else None
            index_r = segment_list_in_r[0] if len(segment_list_in_r) > 0 else None
            data[f"token_l_{i}"] = (
                entity_l.get_segment_label(index_l) if index_l is not None else None
            )
            data[f"token_r_{i}"] = (
                entity_r.get_segment_label(index_r) if index_r is not None else None
            )
            data[f"att_score_pair_{i}"] = attr.score
            data[f"att_score_org_l_{i}"] = None
            data[f"att_score_org_r_{i}"] = None
            data[f"del_match_score_{i}"] = float(score_delete[i + 1])
        dump_data.append(data)

    return dump_data


def dump_pairs(target_dataset_name: str, top_n: int):
    dataset = load_dataset(target_dataset_name, dataset_root_dir)
    for target_matcher_name in matcher_names:
        pair_exps = load_wym_pair_explanation(
            target_dataset_name, pathlib.Path(wym_result_root_dir)
        )
        score_deletions = load_dump_match_score_delete(
            target_dataset_name, target_matcher_name, pathlib.Path(out_root_dir)
        )
        out_dir_path = (
            pathlib.Path(out_root_dir) / target_matcher_name / target_dataset_name
        )
        out_dir_path.mkdir(parents=True, exist_ok=True)

        dump_data = make_dump_data(dataset, pair_exps, top_n, score_deletions)
        pd.DataFrame(dump_data).to_csv(
            out_dir_path / "dump_data.csv", sep="\t", index=False
        )


dump_pairs(dataset_names[TARGET_DATASET_ID], TOP_N)